In [4]:
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import movie_recommender.data as movie_data

# Model selection

## Data Import

In [5]:
importlib.reload(movie_data)

build_movie_report = movie_data.build_movie_report
build_catalog_eda_frames = movie_data.build_catalog_eda_frames
build_user_eda_frames = movie_data.build_user_eda_frames
build_user_report = movie_data.build_user_report
prepare_movielens_frames = movie_data.prepare_movielens_frames
search_movies = movie_data.search_movies

DATA_DIR = Path("../movies-database")

movies        = pd.read_csv(DATA_DIR / "movies.csv")
ratings       = pd.read_csv(DATA_DIR / "ratings.csv")
tags          = pd.read_csv(DATA_DIR / "tags.csv")
links         = pd.read_csv(DATA_DIR / "links.csv")
genome_scores = pd.read_csv(DATA_DIR / "genome-scores.csv")
genome_tags   = pd.read_csv(DATA_DIR / "genome-tags.csv")

print("movies:       ", movies.shape)
print("ratings:      ", ratings.shape)
print("tags:         ", tags.shape)
print("links:        ", links.shape)
print("genome_scores:", genome_scores.shape)
print("genome_tags:  ", genome_tags.shape)

movies:        (62423, 3)
ratings:       (25000095, 4)
tags:          (1093360, 4)
links:         (62423, 3)
genome_scores: (15584448, 3)
genome_tags:   (1128, 2)


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Training spine  (25 M rows, stays lean)
# ─────────────────────────────────────────────────────────────────────────────

# Only merge the lightweight metadata you actually need per-rating.
# Keep genome completely out of this frame.
train = (
    ratings[["userId", "movieId", "rating", "timestamp"]]
    .merge(movies[["movieId", "genres"]], on="movieId", how="left")
)

# Lightweight datetime features for temporal models.
train["year_rated"]  = pd.to_datetime(train["timestamp"], unit="s").dt.year
train["month_rated"] = pd.to_datetime(train["timestamp"], unit="s").dt.month

print(train.shape)
print(train.dtypes)

(25000095, 7)
userId           int64
movieId          int64
rating         float64
timestamp        int64
genres          object
year_rated       int32
month_rated      int32
dtype: object


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Genre one-hot  (62 K × 19 cols)
# ─────────────────────────────────────────────────────────────────────────────
genre_dummies = (
    movies.set_index("movieId")["genres"]
    .str.get_dummies("|")
    .drop(columns=["(no genres listed)"], errors="ignore")
    .add_prefix("genre_")
    .astype("int8")           # saves memory vs int64
)
print(genre_dummies.shape)  # (62_423, 19)

(62423, 19)


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Genome: pivot to WIDE once (13 816 × 1 128 ≈ 120 MB in float32)
#           Store as a separate lookup — never join to ratings.
# ─────────────────────────────────────────────────────────────────────────────
tag_id_to_name = genome_tags.set_index("tagId")["tag"]

genome_wide = (
    genome_scores
    .pivot(index="movieId", columns="tagId", values="relevance")
    .astype("float32")
)
genome_wide.columns = [tag_id_to_name.get(t, str(t)) for t in genome_wide.columns]

print(genome_wide.shape)    # (13_816, 1_128)

(13816, 1128)


In [9]:
# --- BLOCK 0: Integer Encoding & Genre Alignment ---
# 1. Create 0-based integer indices for the embedding layers
user_ids  = np.sort(train["userId"].unique())
movie_ids = np.sort(train["movieId"].unique())

user_to_idx  = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

N_USERS  = len(user_to_idx)
N_MOVIES = len(movie_to_idx)

train["user_idx"]  = train["userId"].map(user_to_idx).astype("int32")
train["movie_idx"] = train["movieId"].map(movie_to_idx).astype("int32")

print(f"Vocabulary -> Users: {N_USERS:,} | Movies: {N_MOVIES:,}")

# 2. Re-align the Genre Matrix so it matches the new integer indices
# (This prevents a NameError since you deleted the old Cell 11!)
idx_to_movie  = {v: k for k, v in movie_to_idx.items()}
all_movie_ids = [idx_to_movie[i] for i in range(N_MOVIES)]

genre_mat = (genre_dummies
             .reindex(all_movie_ids)
             .fillna(0)
             .values
             .astype("float32"))

N_GENRES = genre_mat.shape[1]

Vocabulary -> Users: 162,541 | Movies: 59,047


Block 1: The Leave-One-Out Split & Popularity Weighting
This cell drops the mean-centered ratings. It sorts your data by time, hides the very last movie each user watched to use as the test set, and calculates the inverse-frequency weights to penalize blockbusters.

In [10]:
import numpy as np
import pandas as pd
import random

# --- BLOCK 1: Temporal Sort & Rating Thresholding ---

# 1. Sort by User, then by timestamp
df_ratings_sorted = train.sort_values(['user_idx', 'timestamp']).copy()

# 2. Split by Rating to honor user satisfaction!
# >= 3.5 are "Likes" (Positives). < 3.5 are "Dislikes" (Hard Negatives)
df_likes = df_ratings_sorted[df_ratings_sorted['rating'] >= 3.5].copy()
df_dislikes = df_ratings_sorted[df_ratings_sorted['rating'] < 3.5].copy()

# 3. Extract the LAST "Liked" movie for each user for our Test Set
df_test_pos = df_likes.groupby('user_idx').tail(1).copy()

# 4. The rest of the "Likes" go into the Training Set
df_train_pos = df_likes.drop(df_test_pos.index).copy()

# 5. Set Targets
df_train_pos['target'] = 1.0  # They liked it
df_test_pos['target'] = 1.0   # They liked it
df_dislikes['target'] = 0.0   # Hard Negative: They watched it but didn't like it!

# 6. Calculate Dynamic Weights based on POSITIVE interactions to fix Popularity Bias
movie_counts = df_train_pos['movie_idx'].value_counts()
raw_weights = 1.0 / np.sqrt(movie_counts)
normalized_weights = (raw_weights / raw_weights.mean()).to_dict()

df_train_pos['weight'] = df_train_pos['movie_idx'].map(normalized_weights).fillna(1.0)
df_dislikes['weight'] = df_dislikes['movie_idx'].map(normalized_weights).fillna(1.0)

print(f"Total 'Likes' (Train): {len(df_train_pos)}")
print(f"Total 'Dislikes' (Hard Negatives): {len(df_dislikes)}")

Total 'Likes' (Train): 15467715
Total 'Dislikes' (Hard Negatives): 9369966


Block 2: Negative Sampling for Train & Test
A model can't learn what you want to watch if it doesn't know what you don't want to watch. This cell generates fake "0" interactions for training, and sets up the "1 Positive vs 99 Negatives" test for evaluation.

In [12]:
import random
import pandas as pd
from tqdm import tqdm

# --- BLOCK 2: Fast Negative Sampling with Progress Bar ---

all_movie_list = list(train['movie_idx'].dropna().unique())
user_interacted = df_ratings_sorted.groupby('user_idx')['movie_idx'].apply(set).to_dict()

# Create a "Hard Decoy" pool of the top 2000 most popular movies
top_2000_movies = df_train_pos['movie_idx'].value_counts().head(2000).index.tolist()

# --- TRAIN SET SAMPLED NEGATIVES ---
train_negatives = []

# Use tqdm to wrap the loop — this will show the progress bar!
for _, row in tqdm(df_train_pos.iterrows(), total=len(df_train_pos), desc="Sampling negatives"):
    u = row['user_idx']
    interacted = user_interacted[u]

    negatives = set()
    # Pushed to 7 negatives to force the Neural Network to learn tighter boundaries
    while len(negatives) < 7:
        m_neg = random.choice(all_movie_list)
        if m_neg not in interacted:
            negatives.add(m_neg)

    for m_neg in negatives:
        train_negatives.append({'user_idx': u, 'movie_idx': m_neg, 'target': 0.0, 'weight': 1.0})

# Combine
df_train = pd.concat([
    df_train_pos[['user_idx', 'movie_idx', 'target', 'weight']],
    df_dislikes[['user_idx', 'movie_idx', 'target', 'weight']],
    pd.DataFrame(train_negatives)
])
df_train = df_train.sample(frac=1).reset_index(drop=True)


# --- TEST SET DECOYS (ALL USERS - OPTIMIZED) ---
test_eval_data = []

# Convert the top 2000 list into a set ONCE for incredibly fast math
top_2000_set = set(top_2000_movies)

# Notice we removed the .head()! This will process ALL 162,000 users blazing fast.
for _, row in tqdm(df_test_pos.iterrows(), total=len(df_test_pos), desc="Building ALL test decoys"):
    u = int(row['user_idx'])
    m_pos = int(row['movie_idx'])
    interacted = user_interacted[u]

    # 1. The Set Difference: Instantly find all available decoys
    available_decoys = top_2000_set - interacted

    # 2. The Super User Safety Check
    # If they watched almost everything in the top 2000, just grab whatever is left.
    num_decoys = min(99, len(available_decoys))

    # 3. Instantly sample the exact number we need without a while loop
    decoy_negatives = random.sample(list(available_decoys), num_decoys)

    items_to_score = [m_pos] + decoy_negatives
    test_eval_data.append({'user_idx': u, 'items': items_to_score})

print(f"Final Train Set Size: {len(df_train)}")
print(f"Successfully built decoys for {len(test_eval_data)} users!")
print(f"Ready for HARD HitRate@10 Eval.")

Building ALL test decoys: 100%|██████████| 162414/162414 [01:10<00:00, 2306.24it/s]

Final Train Set Size: 133111686
Successfully built decoys for 162414 users!
Ready for HARD HitRate@10 Eval.


Block 3: The SVD Baseline
We run Matrix Factorization first to set the benchmark. It uses a pure Dot Product.

In [16]:
from scipy.sparse import csr_matrix, vstack
from scipy.sparse.linalg import svds
import numpy as np

print("Aligning Genome Matrix...")
genome_mat = (genome_wide
              .reindex(all_movie_ids)
              .fillna(0)
              .values
              .astype("float32"))

# We give it a massive brain (k=256) to handle the 1,128 Genome tags
K_GRID = [128, 256]
BETA_GRID = [0.1, 0.5] # Genome weight

best_hr = 0
best_params = {}

print("Building the Sparse Matrix 'R'...")
rows = df_train_pos['user_idx'].values
cols = df_train_pos['movie_idx'].values

# THE FIX: Go back to np.ones! Let the model guess blockbusters so it can pass the test!
data = np.ones(len(rows), dtype=np.float32)

R = csr_matrix((data, (rows, cols)), shape=(N_USERS, N_MOVIES), dtype=np.float32)

print("Starting Unchained Hybrid SVD Grid Search...")

for beta in BETA_GRID:
    # Transpose the Genome matrix (we skip genres because Genome already includes them and is 100x better)
    Gen_sparse = csr_matrix((genome_mat * beta).T)

    # THE HYBRID STACK: Users + 1,128 Genome Tags
    R_aug = vstack([R, Gen_sparse], format="csr")

    for k in K_GRID:
        # Note: k=256 will take a minute or two to calculate!
        U, s, Vt = svds(R_aug, k=k)

        # Slice U to only grab the actual User vectors
        U_users = U[:N_USERS, :]
        U_sig = np.dot(U_users, np.diag(s))

        # Evaluate HitRate@10
        hits = 0
        for data_eval in test_eval_data:
            u = data_eval['user_idx']
            items = data_eval['items']

            scores = np.dot(U_sig[u], Vt[:, items])
            top_10_idx = np.argsort(scores)[::-1][:10]

            if 0 in top_10_idx:
                hits += 1

        hr10 = hits / len(test_eval_data)
        print(f"Grid Search -> k={k:3}, genome_w={beta:3.1f} | HitRate@10: {hr10:.4f}")

        if hr10 > best_hr:
            best_hr = hr10
            best_params = {'k': k, 'beta': beta}
            best_Vt = Vt

print(f"\n🏆 Best Params: {best_params} | Best HitRate: {best_hr:.4f}")

Aligning Genome Matrix...
Building the Sparse Matrix 'R'...
Starting Unchained Hybrid SVD Grid Search...
Grid Search -> k=128, genome_w=0.1 | HitRate@10: 0.5049
Grid Search -> k=256, genome_w=0.1 | HitRate@10: 0.4365
Grid Search -> k=128, genome_w=0.5 | HitRate@10: 0.5056
Grid Search -> k=256, genome_w=0.5 | HitRate@10: 0.4372

🏆 Best Params: {'k': 128, 'beta': 0.5} | Best HitRate: 0.5056


Funk SVD

In [15]:
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. THE PYTORCH SVD ARCHITECTURE (Funk SVD)
# ==========================================
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, n_factors=128):
        super().__init__()
        # The exact same embeddings SVD uses mathematically!
        self.u_emb = nn.Embedding(n_users, n_factors)
        self.m_emb = nn.Embedding(n_movies, n_factors)

        # Biases are critical for RMSE (e.g., "This user always rates low")
        self.u_bias = nn.Embedding(n_users, 1)
        self.m_bias = nn.Embedding(n_movies, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

    def forward(self, u, m):
        # Pure Dot Product (No MLP layers!)
        dot = (self.u_emb(u) * self.m_emb(m)).sum(dim=1)
        # Add biases to predict the exact Star Rating
        return dot + self.u_bias(u).squeeze() + self.m_bias(m).squeeze() + self.global_bias

# ==========================================
# 2. THE DATA LOADER (Raw Star Ratings)
# ==========================================
# We use df_ratings_sorted directly so we keep the exact 0.5 to 5.0 stars!
# We drop the test_pos movies so it doesn't cheat.
train_rmse_df = df_ratings_sorted.drop(df_test_pos.index)

class SVD_RMSE_Batcher:
    def __init__(self, df, batch_size=16384):
        self.u = torch.tensor(df['user_idx'].values, dtype=torch.long, device=DEVICE)
        self.m = torch.tensor(df['movie_idx'].values, dtype=torch.long, device=DEVICE)
        self.y = torch.tensor(df['rating'].values, dtype=torch.float32, device=DEVICE)
        self.n, self.bs = len(self.y), batch_size

    def __iter__(self):
        idx = torch.randperm(self.n, device=DEVICE)
        for i in range(0, self.n, self.bs):
            j = idx[i:i + self.bs]
            yield self.u[j], self.m[j], self.y[j]

    # THE FIX: Tell Python exactly how many batches exist so len() works!
    def __len__(self):
        return (self.n + self.bs - 1) // self.bs

train_dl_rmse = SVD_RMSE_Batcher(train_rmse_df)

# ==========================================
# 3. TRAINING & DUAL-EVALUATION
# ==========================================
model_svd = MatrixFactorization(N_USERS, N_MOVIES).to(DEVICE)
optimiser = torch.optim.Adam(model_svd.parameters(), lr=0.005, weight_decay=1e-5)
criterion = nn.MSELoss() # MEAN SQUARED ERROR!

print("Training PyTorch SVD on Exact Star Ratings...")
for epoch in range(1, 11):
    model_svd.train()
    total_loss = 0

    for u, m, y in tqdm(train_dl_rmse, desc=f"Epoch {epoch} [Training]"):
        optimiser.zero_grad()
        preds = model_svd(u, m)
        loss = criterion(preds, y) # Predict the stars!
        loss.backward()
        optimiser.step()
        total_loss += loss.item()

    # --- DUAL EVALUATION ---
    model_svd.eval()

    # Eval 1: RMSE on the Test Set
    test_u = torch.tensor(df_test_pos['user_idx'].values, dtype=torch.long, device=DEVICE)
    test_m = torch.tensor(df_test_pos['movie_idx'].values, dtype=torch.long, device=DEVICE)
    test_y = df_test_pos['rating'].values # True stars

    with torch.no_grad():
        test_preds = model_svd(test_u, test_m).cpu().numpy()
        # Clip predictions between 0.5 and 5.0 stars
        test_preds = np.clip(test_preds, 0.5, 5.0)
        rmse = np.sqrt(mean_squared_error(test_y, test_preds))

    # Eval 2: HitRate@10 (Using predicted stars to rank!)
    hits = 0
    with torch.no_grad():
        for data in tqdm(test_eval_data, desc=f"Epoch {epoch} [HitRate Eval]"):
            num_items = len(data['items'])
            eval_u = torch.tensor([data['user_idx']] * num_items, dtype=torch.long, device=DEVICE)
            eval_m = torch.tensor(data['items'], dtype=torch.long, device=DEVICE)

            # Predict stars for the True Movie + 99 Decoys
            scores = model_svd(eval_u, eval_m).cpu().numpy()
            top_10_idx = np.argsort(scores)[::-1][:10]
            if 0 in top_10_idx:
                hits += 1

    hr10 = hits / len(test_eval_data)

    print(f"✅ Epoch {epoch} | Train MSE: {(total_loss/len(train_dl_rmse)):.4f} | 📉 Test RMSE: {rmse:.4f} Stars | 🚀 HitRate@10: {hr10:.4f}")

Training PyTorch SVD on Exact Star Ratings...


Epoch 1 [HitRate Eval]: 100%|██████████| 162414/162414 [01:13<00:00, 2197.53it/s]


✅ Epoch 1 | Train MSE: 35.1144 | 📉 Test RMSE: 1.5592 Stars | 🚀 HitRate@10: 0.0962


Epoch 2 [HitRate Eval]: 100%|██████████| 162414/162414 [01:12<00:00, 2226.56it/s]


✅ Epoch 2 | Train MSE: 2.2741 | 📉 Test RMSE: 1.0180 Stars | 🚀 HitRate@10: 0.1678


...

✅ Epoch 8 | Train MSE: 0.6983 | 📉 Test RMSE: 0.8117 Stars | 🚀 HitRate@10: 0.2005


Epoch 9 [HitRate Eval]: 100%|██████████| 162414/162414 [00:56<00:00, 2885.11it/s]


✅ Epoch 9 | Train MSE: 0.6919 | 📉 Test RMSE: 0.8146 Stars | 🚀 HitRate@10: 0.2037


Epoch 10 [HitRate Eval]: 100%|██████████| 162414/162414 [00:55<00:00, 2948.03it/s]

✅ Epoch 10 | Train MSE: 0.6877 | 📉 Test RMSE: 0.8143 Stars | 🚀 HitRate@10: 0.2057


In [16]:
import torch

# Save the Funk SVD model weights
torch.save(model_svd.state_dict(), "../docker-app/services/mlengine/funk_svd_weights.pth")
print("✅ Funk SVD Model successfully exported!")

✅ Funk SVD Model successfully exported!


Block 4: The NCF Challenger (BCE & Weights)
Now we train the deep learning model. Notice the custom PyTorch Batcher, the BCEWithLogitsLoss (for predicting probabilities instead of stars), and the dynamic weighting applied to the loss function.

In [12]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.5.1+cu121
CUDA Available:  True
Device Name:     NVIDIA GeForce RTX 4070


In [13]:
import torch
import gc

# Tell Python to collect any unreferenced variables
gc.collect()

# Force PyTorch to empty its "Ghost" cache back to the system
torch.cuda.empty_cache()

# Check how much VRAM is currently being used
allocated = torch.cuda.memory_allocated() / (1024 ** 3)
reserved = torch.cuda.memory_reserved() / (1024 ** 3)
print(f"GPU Memory Allocated: {allocated:.2f} GB")
print(f"GPU Memory Reserved:  {reserved:.2f} GB")

GPU Memory Allocated: 0.00 GB
GPU Memory Reserved:  0.00 GB


In [14]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
genre_tensor_all = torch.tensor(genre_mat, dtype=torch.float32).to(DEVICE) # From your earlier cell

# 1. The GPU Batcher
class GPUBatcherBCE:
    def __init__(self, df, batch_size):
        self.u = torch.tensor(df['user_idx'].values, dtype=torch.long, device=DEVICE)
        self.m = torch.tensor(df['movie_idx'].values, dtype=torch.long, device=DEVICE)
        self.y = torch.tensor(df['target'].values, dtype=torch.float32, device=DEVICE)
        self.w = torch.tensor(df['weight'].values, dtype=torch.float32, device=DEVICE)
        self.g = genre_tensor_all[self.m]
        self.n, self.bs = len(self.y), batch_size

    def __iter__(self):
        idx = torch.randperm(self.n, device=DEVICE)
        for i in range(0, self.n, self.bs):
            j = idx[i:i + self.bs]
            yield self.u[j], self.m[j], self.g[j], self.y[j], self.w[j]


# 2. The Model (No activation on the output layer because BCEWithLogits handles it)
class NeuralCF(nn.Module):
    # Bumped n_factors to 64, kept dropout at 5%
    def __init__(self, n_users, n_movies, n_genres=19, n_factors=64, dropout_rate=0.05):
        super().__init__()
        self.u_gmf = nn.Embedding(n_users, n_factors)
        self.m_gmf = nn.Embedding(n_movies, n_factors)
        self.u_mlp = nn.Embedding(n_users, n_factors)
        self.m_mlp = nn.Embedding(n_movies, n_factors)
        self.genre_proj = nn.Linear(n_genres, n_factors, bias=False)

        # Added a 256-neuron layer to make the network deeper and smarter
        dims = [2 * n_factors + n_genres, 256, 128, 64, 32]

        self.mlp = nn.Sequential(*[
            layer for in_d, out_d in zip(dims, dims[1:])
            for layer in (nn.Linear(in_d, out_d), nn.ReLU(), nn.Dropout(dropout_rate))
        ])
        self.output_layer = nn.Linear(n_factors + dims[-1], 1)

    def forward(self, u, m, g):
        gmf_out = self.u_gmf(u) * (self.m_gmf(m) + self.genre_proj(g))
        mlp_out = self.mlp(torch.cat([self.u_mlp(u), self.m_mlp(m), g], dim=1))
        return self.output_layer(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)

from torch.amp import autocast, GradScaler

# 1. Boost Batch Size for Speed!
train_dl = GPUBatcherBCE(df_train, batch_size=8192)

model_ncf = NeuralCF(N_USERS, N_MOVIES).to(DEVICE)

# 2. Boost Learning Rate so it learns faster
optimiser = torch.optim.Adam(model_ncf.parameters(), lr=0.003)

scheduler = StepLR(optimiser, step_size=6, gamma=0.5)

criterion = nn.BCEWithLogitsLoss(reduction='none')

# 3. Initialize the PyTorch Speed Booster (AMP)
scaler = GradScaler('cuda')

from tqdm import tqdm # <-- Don't forget this import!

print("Training NCF with Turbo Boost...")
# 4. Increase to 15 Epochs!
for epoch in range(1, 16):
    model_ncf.train()
    total_loss = 0

    # 1. ADD TQDM HERE to watch the batches fly by
    for u, m, g, y, w in tqdm(train_dl, desc=f"Epoch {epoch} [Training]"):
        optimiser.zero_grad()

        # Run the forward pass in 16-bit precision for massive speed gains
        with autocast(device_type='cuda'):
            preds = model_ncf(u, m, g)
            loss = criterion(preds, y)
            weighted_loss = torch.mean(loss * w)

        # Scale the loss and backpropagate using the AMP scaler
        scaler.scale(weighted_loss).backward()
        scaler.step(optimiser)
        scaler.update()

        total_loss += weighted_loss.item()

    # --- Evaluation Loop ---
    model_ncf.eval()
    hits = 0
    with torch.no_grad():
        # 2. ADD TQDM HERE to watch the 162,000 users get evaluated
        for data in tqdm(test_eval_data, desc=f"Epoch {epoch} [Evaluating]"):
            # THE FIX: Dynamically match the exact number of items!
            num_items = len(data['items'])
            u = torch.tensor([data['user_idx']] * num_items, dtype=torch.long, device=DEVICE)

            m = torch.tensor(data['items'], dtype=torch.long, device=DEVICE)
            g = genre_tensor_all[m]

            scores = model_ncf(u, m, g).cpu().numpy()
            top_10_idx = np.argsort(scores)[::-1][:10]
            if 0 in top_10_idx:
                hits += 1

    scheduler.step()
    hr10 = hits / len(test_eval_data)
    print(f"✅ Epoch {epoch} Complete | Loss: {total_loss:.4f} | 🚀 NCF HitRate@10: {hr10:.4f} ({(hr10*100):.1f}%)\n")

Training NCF with Turbo Boost...


Epoch 1 [Training]: 4370it [11:41,  6.23it/s]


KeyboardInterrupt: 

In [17]:
# --- FINAL BLOCK: Export for FastAPI ---
import pickle

# 1. Save the new, ranking-optimized model weights
torch.save(model_ncf.state_dict(), "../docker-app/services/mlengine/ncf_weights.pth")


# 2. Save the metadata lookups (including the required genre tensor!)
metadata = {
    "N_USERS": N_USERS,
    "N_MOVIES": N_MOVIES,
    "movie_to_idx": movie_to_idx,
    "idx_to_movie": {v: k for k, v in movie_to_idx.items()},
    "N_GENRES": N_GENRES,
    "all_genres_tensor": genre_tensor_all.cpu()
}

with open("../docker-app/services/mlengine/ncf_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

np.save("../docker-app/services/mlengine/svd_item_matrix.npy", Vt)

print("✅ Model and Metadata successfully exported for FastAPI!")

✅ Model and Metadata successfully exported for FastAPI!


In [14]:
import numpy as np
import torch

# 1. Create a quick lookup dictionary for movie titles
movie_titles = movies.set_index('movieId')['title'].to_dict()

def recommend_vs_showdown(liked_movie_ids, svd_matrix, model_ncf, movie_to_idx, idx_to_movie, top_n=10):
    """
    Creates a synthetic user and generates recommendations from BOTH SVD and NCF models.
    """
    # Convert real MovieLens IDs to our 0-based matrix indices
    liked_indices = []
    for mid in liked_movie_ids:
        if mid in movie_to_idx:
            liked_indices.append(movie_to_idx[mid])
        else:
            print(f"⚠️ Warning: Movie ID {mid} not found in training data.")

    if not liked_indices:
        return "No valid movies found. Try different IDs."

    # ==========================================
    # MODEL 1: SVD (Linear Algebra)
    # ==========================================
    # Sum the item vectors of the movies they liked
    svd_user_vec = np.sum(svd_matrix[:, liked_indices], axis=1)
    svd_scores = np.dot(svd_user_vec, svd_matrix)
    svd_scores[liked_indices] = -np.inf # Hide already liked movies
    svd_top_10 = np.argsort(svd_scores)[::-1][:top_n]

    # ==========================================
    # MODEL 2: NCF (Deep Learning)
    # ==========================================
    model_ncf.eval()
    with torch.no_grad():
        liked_tensor = torch.tensor(liked_indices, dtype=torch.long, device=DEVICE)

        # 1. Create Neural Synthetic User by averaging the liked movie embeddings
        synth_u_gmf = model_ncf.m_gmf(liked_tensor).mean(dim=0)
        synth_u_mlp = model_ncf.m_mlp(liked_tensor).mean(dim=0)

        # 2. Prep all movies for scoring
        all_m = torch.arange(N_MOVIES, device=DEVICE)
        all_g = genre_tensor_all

        # 3. Expand the synthetic user to match the size of the whole catalog
        u_gmf_expanded = synth_u_gmf.unsqueeze(0).expand(N_MOVIES, -1)
        u_mlp_expanded = synth_u_mlp.unsqueeze(0).expand(N_MOVIES, -1)

        # 4. Manual Forward Pass (Bypassing the User Embedding lookup!)
        gmf_out = u_gmf_expanded * (model_ncf.m_gmf(all_m) + model_ncf.genre_proj(all_g))
        mlp_out = model_ncf.mlp(torch.cat([u_mlp_expanded, model_ncf.m_mlp(all_m), all_g], dim=1))
        ncf_scores = model_ncf.output_layer(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)

        # 5. Move to CPU and format
        ncf_scores = ncf_scores.cpu().numpy()
        ncf_scores[liked_indices] = -np.inf # Hide already liked movies
        ncf_top_10 = np.argsort(ncf_scores)[::-1][:top_n]

    # ==========================================
    # PRINT THE SHOWDOWN RESULTS
    # ==========================================
    print("🍿 BASED ON YOUR LIKES, WE RECOMMEND:")
    print(f"{'Rank':<5} | {'SVD Baseline (Linear)':<45} | {'NeuralCF (Deep Learning)':<45}")
    print("-" * 100)

    for rank in range(top_n):
        svd_mid = idx_to_movie[svd_top_10[rank]]
        ncf_mid = idx_to_movie[ncf_top_10[rank]]

        svd_title = movie_titles.get(svd_mid, f"Unknown ID: {svd_mid}")[:42]
        ncf_title = movie_titles.get(ncf_mid, f"Unknown ID: {ncf_mid}")[:42]

        print(f"{rank+1:<5} | {svd_title:<45} | {ncf_title:<45}")


# --- TEST IT OUT ---

# Helper to find IDs by name if you don't know them:
def search_title(keyword):
    return movies[movies['title'].str.contains(keyword, case=False, na=False)][['movieId', 'title']].head(5)

# Example: Let's find the IDs for Matrix and Inception
print("Search Results:")
print(search_title("Ladies First"))
print(search_title("Pulp Fiction"))
print(search_title("Twilight"))
print("\n" + "="*50 + "\n")

# Enter the actual MovieLens IDs you want to test here:
# 296 = Pulp Fiction, 197085 = Ladies First, 1791 = Twilight
my_favorites = [296, 197085, 1791]

# Run the Recommender Showdown!
# (Make sure 'Vt' is your SVD item matrix from the earlier block)
recommend_vs_showdown(my_favorites, Vt, model_ncf, movie_to_idx, idx_to_movie)

Search Results:
       movieId                title
57755   197085  Ladies First (2017)
     movieId                title
292      296  Pulp Fiction (1994)
       movieId                                           title
1712      1791                                 Twilight (1998)
8838     26492                 Twilight Zone: The Movie (1983)
9319     27741  Twilight Samurai, The (Tasogare Seibei) (2002)
11985    56372           Tokyo Twilight (Tôkyô boshoku) (1957)
12826    63992                                 Twilight (2008)


🍿 BASED ON YOUR LIKES, WE RECOMMEND:
Rank  | SVD Baseline (Linear)                         | NeuralCF (Deep Learning)                     
----------------------------------------------------------------------------------------------------
1     | Reservoir Dogs (1992)                         | Charlie's Country (2013)                     
2     | Get Shorty (1995)                             | Family Game, The (Kazoku gêmu) (1983)        
3     | Natural Born